# 《PythAPCS123》單元 13-3：執行時期錯誤（Runtime Error, RE）常見排行榜與崩潰防禦

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者  
**學習目標**：
- 徹底理解執行時期錯誤（Runtime Error, RE）的觸發機制、例外傳播與堆疊追蹤（Traceback）深度閱讀
- 深度排查 APCS 考場最常見的 7 大 RE 致命地雷：`IndexError`、`ValueError`、`KeyError`、`ZeroDivisionError`、`TypeError`、`RecursionError`、`AttributeError` / `UnboundLocalError`
- 建立堅不可摧的「事前條件守門員（LBYL）」防禦體系，杜絕在線上評判系統（Online Judge）遭遇突發崩潰與零分悲劇
- 掌握考場 1 分鐘 RE 極速排查與自測 SOP，培養自主抓蟲與自愈程式碼的實戰能力


### 13.3.1 什麼是例外（Exception）？程式執行中途突發崩潰的觸發機制與 RE 評判本質

在上一單元中，我們深入剖析了直譯器在門口攔截的「語法錯誤（SyntaxError）」。語法錯誤發生在編譯期，整份程式連第 1 行都無法執行。然而，**執行時期錯誤（Runtime Error, 簡稱 RE）**則完全屬於另一個維度：你的程式語法百分之百完全合法，Python 直譯器欣然將其編譯為位元組碼並啟動 Python 虛擬機（PVM）開始逐行執行。程式在開頭可能運作得非常順暢、資料讀取正確、迴圈也順利跑了好幾輪，但當程式執行到某一行特定運算時，碰上了客觀上「在計算機邏輯中無法被執行的非法操作」——例如試圖存取不存在的串列位置、拿字串與數字直接相加、或是將除數設為 0。

當 PVM 遭遇這種無法繼續推進的困境時，直譯器會立刻中斷正常運算流程，在記憶體中建立一個代表該錯誤事件的物件，並由直譯器「拋出（Raise）」此**例外（Exception）**。若程式設計師未事先設定任何防護措施，這個例外就會如滾雪球般沿著函式呼叫鏈由內向外傳播，這就是所謂的「例外冒泡（Exception Bubbling）」。一旦例外傳播到最頂層模組仍無人攔截，PVM 便會瞬間崩潰暴斃，印出整片紅色的堆疊追蹤訊息（Traceback）並終止執行。

在 APCS 考場與各類線上評判系統（Online Judge）中，評判伺服器會監聽 Python 程式的退出代碼（Exit Code）。正常執行完畢的程式退出代碼為 0；但凡拋出任何未捕捉的例外，Python 直譯器都會以非零代碼（如 1）異常終止。評判系統只要偵測到非零結束狀態，就會毫不留情地給出冷冰冰的 **`RE (Runtime Error)`** 評判結果！特別致命的是：如果你的程式在公開的基礎範例測資運作良好，卻在隱藏測資遇到「空輸入」或「邊界極端值」觸發未預期的例外，整道題目的分數就會瞬間化為烏有。因此，學會看懂 RE 的堆疊追蹤並建立事前防禦意識，是晉升 APCS 高分選手的必經之路！


In [ ]:
# 13.3.1 程式碼演示：透視例外觸發、例外冒泡與 Traceback 堆疊結構
import traceback
import sys

def level_three(arr, idx):
    print("      [進入 level_three] 準備存取陣列...")
    # 潛在危險操作：若 idx 超出範圍將在此處引爆例外
    return arr[idx]

def level_two(arr, idx):
    print("    [進入 level_two] 呼叫 level_three...")
    val = level_three(arr, idx)
    return val * 2

def level_one(arr, idx):
    print("  [進入 level_one] 呼叫 level_two...")
    return level_two(arr, idx)

print("=== 案例 1: 正常安全調用（所有函式依序返回）===")
result = level_one([10, 20, 30], 1)
print(f"  --> 安全執行完成，計算結果: {result}\n")

print("=== 案例 2: 觸發例外時的「呼叫堆疊（Call Stack）」冒泡追蹤 ===")
try:
    # 故意傳入不存在的索引 99，引爆 IndexError
    level_one([10, 20, 30], 99)
except IndexError as e:
    print("\n  ⚠️ [直譯器捕捉到例外！]")
    print(f"  例外型別: {type(e).__name__}")
    print(f"  例外訊息: {e}")
    print("\n  --- 呼叫堆疊追蹤（Traceback 由外而內展示事故現場）---")
    tb_lines = traceback.format_exc().splitlines()
    for line in tb_lines:
        print(f"  | {line}")
    print("  💡 閱讀心法：看 Traceback 請先看最底部的錯誤型態，再看倒數第二行的出事函式與行號！")


### 13.3.1 語法重點回顧與核心觀念提煉

1. **RE 的本質是執行期的未預期崩潰**：語法完全通過編譯，但在動態運算過程中遭遇非法操作（如越界、轉型失敗、除零）。
2. **例外冒泡機制**：深層函式出事時，若沒有就地攔截，錯誤會一路向上層呼叫者傳播，直到最外層導致直譯器異常中斷。
3. **Traceback 閱讀二字訣「倒著看」**：
   - 第一眼：看最末行（`IndexError: list index out of range`），立刻確認是什麼類型的運算違規。
   - 第二眼：看倒數第二個 `File "...", line X, in ...`，立刻鎖定引爆災難的最初事發程式碼行號！


In [ ]:
# 13.3.1 學生實作練習：萬能安全函式執行診斷器
# 任務說明：實作 safe_execute_and_diagnose(func, *args)
# 1. 嘗試執行 func(*args)
# 2. 若函式執行成功且未發生任何崩潰，回傳元組 ("SUCCESS", 運算回傳值)
# 3. 若函式執行過程中引發任何例外（Exception），捕捉該例外並回傳 ("RE", 例外型別名稱字串)

def safe_execute_and_diagnose(func, *args):
    # 請在此處使用 try-except 實作安全防護與診斷
    try:
        res = func(*args)
        return ("SUCCESS", res)
    except Exception as e:
        return ("RE", type(e).__name__)

# 測試用例
def divide(a, b): return a / b
def get_elem(lst, i): return lst[i]

print("測試正常相除:", safe_execute_and_diagnose(divide, 10, 2))
print("測試除以零崩潰:", safe_execute_and_diagnose(divide, 10, 0))
print("測試索引越界崩潰:", safe_execute_and_diagnose(get_elem, [1, 2, 3], 10))


In [ ]:
# 13.3.1 單元測試驗證
assert safe_execute_and_diagnose(lambda x: x + 10, 5) == ("SUCCESS", 15)
assert safe_execute_and_diagnose(lambda a, b: a / b, 10, 0) == ("RE", "ZeroDivisionError")
assert safe_execute_and_diagnose(lambda lst, i: lst[i], [1, 2], 5) == ("RE", "IndexError")
assert safe_execute_and_diagnose(lambda s: int(s), "abc") == ("RE", "ValueError")
print("🎉 13.3.1 所有測試通過！成功建立例外傳播與診斷核心觀念！")


### 13.3.2 RE 排行榜榜首：`IndexError: list index out of range` 越界地雷深度排查

在所有初學者向線上評判系統送出 Python 程式碼所遭遇的 RE 之中，**`IndexError: list index out of range`**（串列索引超出範圍）以超過 40% 的壓倒性比例穩坐排行榜榜首！這個錯誤的產生，根源於電腦程式語言嚴格的記憶體定址規則與人類日常直覺的衝突。

考場最高頻引爆 `IndexError` 的四大情境：
1. **0-based 偏移量誤解**：長度為 $N$ 的串列，合法正向索引僅為 $0 \sim N-1$。初學者常下意識存取 `lst[N]`（例如長度為 3 的串列存取 `lst[3]`），直譯器立刻丟出越界崩潰！
2. **存取空串列（Empty List Access）**：當串列長度為 0（例如 `lst = []`）時，它**不存在任何合法索引**，哪怕試圖存取第 0 個元素 `lst[0]` 都會當場引爆 `IndexError`。在考場讀取篩選後的資料時，若沒有任何資料符合條件導致串列為空，後續直接寫 `lst[0]` 是極其常見的隱形丟分陷阱！
3. **相鄰元素探測溢出（Neighbor Inspection Overflow）**：在陣列走訪題目中，常常需要檢查「當前元素與下一個元素」的關係（如 `if a[i] < a[i+1]:`）。如果迴圈範圍寫成 `for i in range(len(a)):`，當迴圈跑到最後一個元素 $i = N-1$ 時，$i+1$ 就會變成 $N$，直接撞牆崩潰！
4. **負向索引過度回溯**：Python 支援負向索引（$-1 \sim -N$ 代表倒數第 1 到倒數第 $N$ 個元素）。但若索引小於 $-N$（例如長度為 3 卻存取 `lst[-4]`），同樣會引爆 `IndexError`。

🛡️ **考場防守第一心法：條件式長度守門員（LBYL）與安全切片（Safe Slicing）**
- **長度守門員**：存取前永遠養成檢查習慣：`if 0 <= i < len(a): return a[i]`。
- **神奇切片特性**：在 Python 中，單一元素索引超出範圍會丟出例外，但**範圍切片（Slicing）永遠不會報錯**！例如長度為 3 的串列，執行 `lst[10:20]` 只會靜默回傳空串列 `[]` 而不會引發崩潰。善用這個特性可以在特殊搜尋場景巧妙避開越界風險。


In [ ]:
# 13.3.2 程式碼演示：四大越界地雷重現與兩種安全防禦心法
print("--- 雷區 1: 長度為 N 誤取 lst[N] 與空串列存取 ---")
nums = [10, 20, 30]
try:
    print(nums[3]) # 長度為 3，最大索引只有 2！
except IndexError as e:
    print(f"  [捕捉成功 1] nums[3] 觸發: {e}")

empty_list = []
try:
    print(empty_list[0])
except IndexError as e:
    print(f"  [捕捉成功 2] empty_list[0] 觸發: {e}")

print("\n--- 雷區 2: 相鄰探測 a[i+1] 尾部溢出災難 ---")
def find_increasing_pairs_buggy(arr):
    # 錯誤寫法：range(len(arr)) 跑到最後一格會引爆 a[i+1]
    count = 0
    for i in range(len(arr)):
        # 當 i == len(arr) - 1 時，i + 1 越界！
        try:
            if arr[i] < arr[i + 1]:
                count += 1
        except IndexError:
            print(f"  ⚠️ 當走訪至索引 i={i} 時，arr[i+1] 慘遭 IndexError 崩潰！")
            return -1
    return count

find_increasing_pairs_buggy([1, 5, 3, 8])

print("\n--- 防禦技術：安全切片 (Safe Slicing) 與邊界保護 ---")
# 單一元素存取超出範圍會報錯：nums[99] -> Crash!
# 但切片超出範圍永不崩潰：
safe_slice = nums[99:105]
print(f"  nums[99:105] 切片回傳: {safe_slice} (安全回傳空串列，絕不引發 RE！)")


### 13.3.2 語法重點回顧與核心觀念提煉

1. **合法索引牢記公式**：
   - 正向：$0 \le \text{index} < \text{len}(a)$
   - 負向：$-\text{len}(a) \le \text{index} \le -1$
2. **相鄰走訪迴圈界限**：
   - 探測右鄰居 `a[i+1]`：迴圈務必寫 `for i in range(len(a) - 1):`！
   - 探測左鄰居 `a[i-1]`：迴圈務必從 1 開始 `for i in range(1, len(a)):`！
3. **空串列優先判斷**：存取 `lst[0]` 前，務必先確認 `if lst:` 或 `if len(lst) > 0:`。


In [ ]:
# 13.3.2 學生實作練習：安全索引存取器與相鄰元素差值計算
# 任務說明：實作兩個安全函式
# 1. safe_get(lst, idx, default=None)：
#    安全取出 lst[idx]，支援正負索引。若 idx 越界或 lst 為空，安全回傳 default，絕不引發 IndexError！
# 2. safe_adjacent_diffs(lst)：
#    計算串列中所有相鄰兩數的差值 lst[i+1] - lst[i]，回傳差值串列。
#    若長度小於 2，安全回傳空串列 []，絕不越界！

def safe_get(lst, idx, default=None):
    # 請在此處實作邊界守門員
    n = len(lst)
    if -n <= idx < n:
        return lst[idx]
    return default

def safe_adjacent_diffs(lst):
    # 請在此處使用正確的迴圈界限計算相鄰差值
    diffs = []
    for i in range(len(lst) - 1):
        diffs.append(lst[i + 1] - lst[i])
    return diffs

# 測試用例
sample = [10, 25, 40]
print("安全存取合法:", safe_get(sample, 1))
print("安全存取越界:", safe_get(sample, 5, -1))
print("安全存取負數越界:", safe_get(sample, -5, "OUT"))
print("計算相鄰差值:", safe_adjacent_diffs([10, 20, 35, 30]))
print("計算長度為1之相鄰差值:", safe_adjacent_diffs([99]))


In [ ]:
# 13.3.2 單元測試驗證
assert safe_get([1, 2, 3], 0) == 1
assert safe_get([1, 2, 3], 2) == 3
assert safe_get([1, 2, 3], 3) == None
assert safe_get([1, 2, 3], -1) == 3
assert safe_get([1, 2, 3], -4, 999) == 999
assert safe_get([], 0, "EMPTY") == "EMPTY"

assert safe_adjacent_diffs([10, 20, 35]) == [10, 15]
assert safe_adjacent_diffs([5]) == []
assert safe_adjacent_diffs([]) == []
print("🎉 13.3.2 所有測試通過！徹底征服 IndexError 越界陷阱！")


### 13.3.3 RE 排行榜第二名：`ValueError` 數值轉換、格式與解包失敗深度剖析

在 APCS 考場的執行時期錯誤排行榜中，**`ValueError`** 緊追在 IndexError 之後位列第二。
什麼是 `ValueError`？它的定義是：「直譯器收到的引數型態是正確的（例如傳入的是字串型態），但字串內部所包含的具體內容數值卻不合法，無法被該函式所接受或解析。」

考場中最高頻率引爆 `ValueError` 的三大重災區：
1. **強制型態轉換失敗（Type Casting Failure）**：
   - 考場中最常見的輸入處理是 `int(token)`。如果輸入資料中夾雜了英文字母、特殊符號或是空字串，例如 `int("hello")` 或 `int("")`，Python 無法將其辨識為十進位整數，就會直接拋出 `ValueError: invalid literal for int() with base 10`。
   - 特別容易被忽略的是**浮點數字串直接轉整數**！例如 `int("3.14")` 會引爆 `ValueError`！因為 `int()` 函式只接受整數格式字串，正確做法必須先經過 `float("3.14")` 轉為浮點數，再轉成 `int()`。
2. **多變數解包數量不對稱（Unpacking Mismatch）**：
   - 初學者在讀取單行多個數字時，極度習慣寫出：`a, b = map(int, input().split())`。
   - 這行程式碼背後隱含著極其嚴格的假設：「該行輸入必定剛好包含兩個數值」。如果線上評判系統送出的某筆極端測試資料中只有一個數字、或者考生多按了一個數字包含三個數值，直譯器就會當場引爆：
     - 若資料太少：`ValueError: not enough values to unpack (expected 2, got 1)`
     - 若資料太多：`ValueError: too many values to unpack (expected 2)`
3. **串列搜尋查無此人（`list.index()` Failure）**：
   - 許多學生喜歡使用 `lst.index(target)` 來查找某個數值在串列中的位置。然而，在 Python 的官方實作中，如果 `target` 根本不在 `lst` 內，該方法**不會回傳 -1，而是直接丟出致命的 `ValueError: ... is not in list`**！這與 C++、Java 或 JavaScript 的慣例截然不同，常常成為考生在考場上猝不及防的 RE 伏兵。

🛡️ **防禦心法**：
- 在呼叫 `lst.index(x)` 前，必須先使用成員運算子 `if x in lst:` 築起護城河。
- 在解包前，先將 `input().split()` 的結果存為清單，檢查 `len(tokens) >= 2` 後再行提取！


In [ ]:
# 13.3.3 程式碼演示：三大 ValueError 翻車現場與安全解包/查詢模式
print("--- 雷區 1: int() 強轉型失敗現場 ---")
test_literals = ["123", "  456  ", "3.14", "abc", ""]
for s in test_literals:
    try:
        val = int(s)
        print(f"  [成功解析] {repr(s)} -> {val}")
    except ValueError as e:
        print(f"  ❌ [ValueError] 無法將 {repr(s)} 轉為整數: {e}")

print("\n--- 雷區 2: 解包個數不對稱 (too many / not enough) ---")
test_inputs = ["10 20", "10", "10 20 30"]
for s in test_inputs:
    tokens = s.split()
    try:
        a, b = tokens # 嚴格要求剛好 2 個
        print(f"  [解包成功] a={a}, b={b}")
    except ValueError as e:
        print(f"  ❌ [解包崩潰] 輸入 {repr(s)} (個數={len(tokens)}): {e}")

print("\n--- 雷區 3: list.index() 查無數值崩潰現場 ---")
colors = ["red", "blue", "green"]
target = "yellow"
try:
    idx = colors.index(target)
except ValueError as e:
    print(f"  ❌ colors.index('{target}') 拋出例外: {e}")
    # 安全寫法演示：
    idx = colors.index(target) if target in colors else -1
    print(f"  🛡️ 安全查詢結果: target 索引為 {idx} (查無此人安全回傳 -1)")


### 13.3.3 語法重點回顧與核心觀念提煉

1. **`int()` 轉換防身守則**：
   - 帶小數點字串（如 `"3.14"`）轉整數：必須寫成 `int(float("3.14"))`。
   - 使用 `.strip()` 濾除頭尾殘留的空白與換行字元。
2. **解包前先檢查長度**：
   - 避免直接 `a, b = line.split()`。建議先 `tokens = line.split()`，確認 `if len(tokens) == 2:` 再行賦值。
3. **`list.index()` 必備守門員**：
   - 養成在 `lst.index(x)` 前先做 `if x in lst:` 條件防禦的肌肉記憶。


In [ ]:
# 13.3.3 學生實作練習：安全整數剖析器與安全索引查找
# 任務說明：實作兩個抗崩潰函式
# 1. safe_parse_int(token, default=0)：
#    嘗試將 token 轉為整數。若為有效整數（包含字串前後有空白、或是帶小數的浮點字串取整數）回傳整數值；
#    若包含無效英文字母、純符號或空字串，安全回傳 default，絕不引發 ValueError！
# 2. safe_find_index(lst, target)：
#    查找 target 在 lst 中的索引位置。若存在回傳其第 1 次出現的 0-based 索引；
#    若不存在，安全回傳 -1，絕不拋出 ValueError！

def safe_parse_int(token: str, default: int = 0) -> int:
    # 請在此處實作安全數值剖析
    try:
        return int(token)
    except ValueError:
        try:
            return int(float(token))
        except ValueError:
            return default

def safe_find_index(lst: list, target) -> int:
    # 請在此處實作安全索引查找
    if target in lst:
        return lst.index(target)
    return -1

# 測試用例
print("剖析純整數:", safe_parse_int(" 123 "))
print("剖析浮點字串:", safe_parse_int("45.67"))
print("剖析無效字母:", safe_parse_int("abc", -1))
print("查找存在元素:", safe_find_index([10, 20, 30], 20))
print("查找不存在元素:", safe_find_index([10, 20, 30], 99))


In [ ]:
# 13.3.3 單元測試驗證
assert safe_parse_int("100") == 100
assert safe_parse_int("  -50  ") == -50
assert safe_parse_int("3.99") == 3
assert safe_parse_int("invalid", 0) == 0
assert safe_parse_int("", -999) == -999

assert safe_find_index(["apple", "banana"], "banana") == 1
assert safe_find_index(["apple", "banana"], "cherry") == -1
assert safe_find_index([], "apple") == -1
print("🎉 13.3.3 所有測試通過！徹底化解 ValueError 數值轉型與查找陷阱！")


### 13.3.4 RE 排行榜第三名：`KeyError` 字典查無此鍵與集合移除失敗

在 APCS 實作第二題與第三題中，雜湊表（Hash Table，即 Python 中的 `dict` 字典與 `set` 集合）是不可或缺的資料結構利器。它們擁有 $O(1)$ 的極速查找優勢，但如果使用方式缺乏邊界意識，**`KeyError`** 就會成為摧毀程式的罪魁禍首。

考場最常見引發 `KeyError` 的三大場景：
1. **字典中直接存取未登錄的鍵（Direct Non-existent Key Access）**：
   - 字典的語法 `val = my_dict[key]` 是建立在「該鍵必定存在」的強烈假設下。一旦傳入的 `key` 尚未被加入字典，Python 直譯器不會像其他某些寬鬆語言那樣靜默回傳 `None` 或空值，而是會立即引爆 `KeyError: '...'` 並當場崩潰！
2. **統計次數時的「未初始化即累加」陷阱**：
   - 這是初學者在實作字頻統計、分類累計或圖論度數統計時最常犯的手滑錯誤：直接寫 `counter[word] += 1`。在該單字第 1 次出現時，由於 `counter[word]` 根本還沒被賦值，直譯器在嘗試讀取它的舊值來加 1 時，瞬間觸發 `KeyError` 陣亡！
3. **集合的 `set.remove(x)` 剛性移除陷阱**：
   - 在集合操作中，很多學生習慣使用 `s.remove(x)`。但 Python 的 `remove` 方法具有剛性契約：若 `x` 不在集合內，直譯器會拋出 `KeyError`！
   - 相比之下，Python 提供了另一個更為優雅的內建方法——**`set.discard(x)`**！若元素存在則將其移除；若元素原本就不在集合中，則靜默略過，**保證絕不引發 KeyError**！

🛡️ **防禦心法：三大防禦武器告別 KeyError**：
1. **成員運算子守門員**：`if key in my_dict: val = my_dict[key]`。
2. **安全取值方法 `dict.get(key, default)`**：若鍵存在回傳其對應的值；若鍵不存在，安全回傳指定的 `default`（預設為 `None`），永不崩潰！
3. **標準庫神器 `collections.defaultdict`**：在建立計數器時，使用 `defaultdict(int)`，遇到新鍵自動初始化為預設值 0，徹底告別累加前的初始化繁瑣判斷！


In [ ]:
# 13.3.4 程式碼演示：KeyError 翻車、set.remove 陷阱與三大防禦武器
print("--- 雷區 1: 直接中括號取值與累加引爆 KeyError ---")
scores = {"Alice": 95, "Bob": 80}
try:
    print(scores["Charlie"]) # 查無 Charlie
except KeyError as e:
    print(f"  ❌ 直接存取 scores['Charlie'] 觸發 KeyError: {e}")

word_counts = {}
try:
    word_counts["apple"] += 1 # 尚未初始化就加 1！
except KeyError as e:
    print(f"  ❌ 未初始化直接累加 word_counts['apple'] += 1 觸發 KeyError: {e}")

print("\n--- 雷區 2: set.remove() vs set.discard() 剛性與彈性對比 ---")
my_set = {1, 2, 3}
try:
    my_set.remove(99) # 99 不在集合中，引爆 KeyError！
except KeyError as e:
    print(f"  ❌ my_set.remove(99) 觸發 KeyError: {e}")

# 彈性安全替代方案：discard
my_set.discard(99) # 安全略過，完全不報錯！
print("  🛡️ my_set.discard(99) 執行成功，集合內容保持原樣且不崩潰！")

print("\n--- 防禦武器展示：dict.get() 與 defaultdict ---")
# 武器 1: dict.get()
charlie_score = scores.get("Charlie", 0)
print(f"  🛡️ scores.get('Charlie', 0) 回傳安全值: {charlie_score}")

# 武器 2: defaultdict
from collections import defaultdict
safe_counter = defaultdict(int)
safe_counter["banana"] += 1 # 自動建立預設值 0 後再加 1，永不報錯！
print(f"  🛡️ safe_counter['banana'] 累加後結果: {safe_counter['banana']}")


### 13.3.4 語法重點回顧與核心觀念提煉

1. **查字典必備反射動作**：
   - 只要不確定鍵是否存在，一律使用 `my_dict.get(key, default)` 取代中括號 `my_dict[key]`。
2. **集合刪除首選 `discard()`**：
   - 考場中除掉集合元素時，無腦選用 `s.discard(x)`，徹底免疫 `KeyError`。
3. **字頻與分組神器 `defaultdict`**：
   - 計數用 `defaultdict(int)`，群組分類用 `defaultdict(list)`，程式碼簡潔且絕不崩潰。


In [ ]:
# 13.3.4 學生實作練習：抗崩潰字頻計數與安全字典查詢器
# 任務說明：實作兩個函式
# 1. safe_count_frequencies(words)：
#    傳入字串清單 words，統計每個單字出現的次數並回傳一般字典。
#    要求：程式碼中嚴格禁止出現 KeyError 崩潰，必須妥善處理首次出現的單字！
# 2. safe_query_user(db, username, default_role="guest")：
#    查詢 db 字典中 username 對應的角色名稱。
#    若查無該使用者，安全回傳 default_role，絕不引發 KeyError！

def safe_count_frequencies(words: list) -> dict:
    counts = {}
    # 請在此處使用 dict.get 或條件判斷實作安全計數
    for w in words:
        counts[w] = counts.get(w, 0) + 1
    return counts

def safe_query_user(db: dict, username: str, default_role: str = "guest") -> str:
    # 請在此處實作安全字典查詢
    return db.get(username, default_role)

# 測試用例
sample_words = ["apple", "banana", "apple", "cherry", "banana", "apple"]
print("字頻統計結果:", safe_count_frequencies(sample_words))
user_db = {"admin": "root", "johnny": "developer"}
print("查詢存在用戶:", safe_query_user(user_db, "admin"))
print("查詢不存在用戶:", safe_query_user(user_db, "unknown_user"))


In [ ]:
# 13.3.4 單元測試驗證
freq = safe_count_frequencies(["a", "b", "a", "c", "b", "a"])
assert freq == {"a": 3, "b": 2, "c": 1}
assert safe_count_frequencies([]) == {}

db = {"alice": "manager", "bob": "staff"}
assert safe_query_user(db, "alice") == "manager"
assert safe_query_user(db, "nobody") == "guest"
assert safe_query_user(db, "nobody", "anonymous") == "anonymous"
print("🎉 13.3.4 所有測試通過！徹底防禦 KeyError 字典查無此鍵地雷！")


### 13.3.5 RE 排行榜第四名：`ZeroDivisionError` 除以零與模除零陷阱

在數學界中，「以零為除數」屬於未定義（Undefined）的無效運算；而在電腦處理器與直譯器層級，任何試圖除以零的操作都會直接引爆硬體或直譯器的異常中斷。在 Python 中，這種操作會明確觸發 **`ZeroDivisionError: division by zero`**（或 `integer division or modulo by zero`）。

考場中會引爆 `ZeroDivisionError` 的三大運算子與隱形場景：
1. **三大危險運算子**：
   - 浮點除法：`a / b`（當 `b == 0` 時）
   - 整數整除：`a // b`（當 `b == 0` 時）
   - 取餘數模除：`a % b`（當 `b == 0` 時）——特別注意：很多學生以為只有除號會出事，實際上**取餘數運算 `%` 若右側為 0 同樣會當場暴斃**！
2. **計算平均值時的「空資料集」陷阱**：
   - 考場計算平均值是家常便飯：`avg = sum(scores) / len(scores)`。
   - 命題老師在設計隱藏測試資料時，極常故意給出「經過條件篩選後符合條件的資料個數為 0」的極端測資！此時 `len(scores)` 恰好等於 0，整份程式連跑都不用跑，直接在最後印出平均時轟然倒地！
3. **迴圈動態變數或步長縮減至 0**：
   - 在模擬演算法（如資源分配、輾轉相除法手刻、步長縮減）時，分母變數可能隨著迭代逐步遞減，最終在最後一輪變成 0，若未設好收斂終止條件，就會在最後一刻飲恨。

🛡️ **防禦心法：分母非零守門員**
在進行任何帶有 `/`、`//` 或 `%` 的運算前，永遠將分母變數納入先決條件檢查：
`result = (total / count) if count != 0 else default_value`。
在平均值計算中，若資料長度為 0，應直接特判回傳 0.0 或依題目規範輸出特定標記。


In [ ]:
# 13.3.5 程式碼演示：三大除零運算子崩潰重現與空資料集平均防禦
print("--- 雷區 1: 三大運算子除以零翻車現場 ---")
operators_test = [
    ("浮點除法 10 / 0", lambda: 10 / 0),
    ("整數整除 10 // 0", lambda: 10 // 0),
    ("取餘數模除 10 % 0", lambda: 10 % 0)
]
for desc, op in operators_test:
    try:
        op()
    except ZeroDivisionError as e:
        print(f"  ❌ [{desc}] 觸發 ZeroDivisionError: {e}")

print("\n--- 雷區 2: 考場計算平均值空資料集慘劇 ---")
def calculate_average_buggy(numbers):
    # 隱形漏洞：完全沒考慮 numbers 為空串列的情況！
    return sum(numbers) / len(numbers)

try:
    print("正常計算:", calculate_average_buggy([80, 90, 100]))
    print("空串列計算:")
    calculate_average_buggy([])
except ZeroDivisionError as e:
    print(f"  ❌ 空串列計算平均時引爆 ZeroDivisionError: {e}")

print("\n--- 防禦技術：條件表達式安全除法 ---")
def calculate_average_safe(numbers, default=0.0):
    # 建立長度非零守門員
    if not numbers:
        return default
    return sum(numbers) / len(numbers)

print("  🛡️ 安全計算結果:", calculate_average_safe([]))


### 13.3.5 語法重點回顧與核心觀念提煉

1. **除數三兄弟全部受害**：`/`、`//`、`%` 三者遇到分母為 0 一視同仁全部引爆 `ZeroDivisionError`。
2. **平均值公式必備前置守衛**：
   - 看到 `len()` 出現在分母，大腦立刻亮起紅燈警報！
   - 永遠加上 `if len(arr) > 0:` 或使用三元運算子 `sum(arr) / len(arr) if arr else 0.0`。


In [ ]:
# 13.3.5 學生實作練習：安全除法與分組平均計算器
# 任務說明：實作兩個抗崩潰數值運算函式
# 1. safe_divide(numerator, denominator, default=0.0)：
#    計算 numerator / denominator。若分母 denominator 為 0，安全回傳 default，絕不引發 ZeroDivisionError！
# 2. safe_modulo(a, b, default=-1)：
#    計算 a % b。若 b 為 0，安全回傳 default，絕不引發崩潰！

def safe_divide(numerator: float, denominator: float, default: float = 0.0) -> float:
    # 請在此處實作分母守門員
    if denominator == 0:
        return default
    return numerator / denominator

def safe_modulo(a: int, b: int, default: int = -1) -> int:
    # 請在此處實作模除安全檢查
    if b == 0:
        return default
    return a % b

# 測試用例
print("正常相除:", safe_divide(10, 2))
print("除以零防禦:", safe_divide(10, 0))
print("正常取餘:", safe_modulo(10, 3))
print("模除零防禦:", safe_modulo(10, 0))


In [ ]:
# 13.3.5 單元測試驗證
assert safe_divide(15, 3) == 5.0
assert safe_divide(10, 0) == 0.0
assert safe_divide(10, 0, -1.0) == -1.0
assert safe_modulo(17, 5) == 2
assert safe_modulo(10, 0) == -1
assert safe_modulo(10, 0, 999) == 999
print("🎉 13.3.5 所有測試通過！徹底告別 ZeroDivisionError 除以零災難！")


### 13.3.6 RE 排行榜第五名：`TypeError` 型態衝突、NoneType 存取與不可呼叫物件

Python 是一門具備「強型別（Strongly Typed）」特性的語言。直譯器絕對不會像 JavaScript 那樣，自動將字串 `"100"` 與整數 `20` 隱式轉換並拼接在一起。當我們嘗試對不相容的資料型態套用某個運算子、或是對不可進行該行為的物件發動呼叫時，直譯器就會毫不猶豫地亮出 **`TypeError`** 紅牌！

考場三大極高頻 `TypeError` 翻車事故：
1. **字串與數字直接串接（Unsupported Operand Types）**：
   - 考場輸出時最常手滑寫出：`print("Answer is: " + ans)`（其中 `ans` 為整數 42）。直譯器會拋出：`TypeError: can only concatenate str (not "int") to str`。
   - 正確做法是使用 f-string：`print(f"Answer is: {ans}")` 或顯式轉型 `str(ans)`。
2. **原地修改方法（In-place Mutation）引爆「NoneType 幽靈」**：
   - 這是 APCS 初學者長年最痛的「自毀武功」大坑！
   - Python 中有許多串列方法是**原地直接修改原始清單，其本身回傳值為 `None`**！
   - 例如初學者常寫出：`nums = nums.sort()` 或是 `res = lst.append(5)`。
   - 這行代碼一跑完，`nums` 或 `res` 就變成了 `None`！後續當你興高采烈地執行 `nums[0]` 或 `nums.append(10)` 時，直譯器當場引爆：
     - `TypeError: 'NoneType' object is not subscriptable`（None 無法取索引）
     - `AttributeError: 'NoneType' object has no attribute 'append'`
3. **變數名稱覆蓋內建函式引爆「不可呼叫（Not Callable）」**：
   - 初學者為了方便，把累加總和命名為 `sum = 0`，或者把清單命名為 `list = [1, 2, 3]`。
   - 此時全域命名空間中的內建函式 `sum()` 和 `list()` 已經被你的整數或串列污染覆蓋！當後續你想呼叫 `total = sum([1, 2, 3])` 時，Python 會發現 `sum` 現在是個整數 0，整數當然不能被當作函式呼叫，當場丟出致命的：
     `TypeError: 'int' object is not callable`！

🛡️ **防禦心法**：
- 永遠牢記：`lst.sort()` 原地修改不回傳（回傳 None），想要新串列必須用 `sorted(lst)`！
- 永遠嚴禁將變數命名為 Python 內建關鍵函式名（如 `sum`, `list`, `max`, `min`, `len`, `dict`, `set`, `str`, `int`）。


In [ ]:
# 13.3.6 程式碼演示：三大 TypeError 翻車現場與 NoneType 幽靈剖析
print("--- 雷區 1: 字串與數值直接用 + 串接 ---")
ans = 42
try:
    bad_msg = "Result: " + ans
except TypeError as e:
    print(f"  ❌ 直接相加引爆 TypeError: {e}")
    # 正解：f-string
    print(f"  🛡️ 正確做法 f-string: {f'Result: {ans}'}")

print("\n--- 雷區 2: 原地排序 nums = nums.sort() 導致 NoneType 暴斃 ---")
raw_data = [3, 1, 4, 1, 5]
# 致命手滑：
broken_data = raw_data.sort() # sort() 回傳 None！
print(f"  broken_data 的實際內容是: {broken_data} (變成 None 了！)")
try:
    print(broken_data[0]) # 試圖存取 None 的第 0 格
except TypeError as e:
    print(f"  ❌ 存取 NoneType 索引引爆: {e}")

# 正確做法對比：
correct_data = sorted([3, 1, 4, 1, 5])
print(f"  🛡️ sorted() 正確回傳排序後新串列: {correct_data}")

print("\n--- 雷區 3: 變數遮蔽內建函式引爆 Not Callable ---")
# 示範遮蔽
original_list = [10, 20, 30]
try:
    # 模擬考生將變數命名為 sum
    fake_sum = 0
    # 假設考生宣告了 sum = fake_sum
    # 後續嘗試呼叫 fake_sum(...)
    fake_sum([10, 20])
except TypeError as e:
    print(f"  ❌ 呼叫整數引爆 TypeError: {e}")


### 13.3.6 語法重點回顧與核心觀念提煉

1. **字串格式化唯一首選 `f"{var}"`**：全面淘汰 `+` 串接字串與變數，徹底免疫 `TypeError`。
2. **區分 In-place vs Return New**：
   - 原地修改（回傳 None）：`lst.sort()`, `lst.reverse()`, `lst.append()`, `lst.extend()`。絕不可賦值回原變數！
   - 回傳新物件：`sorted(lst)`, `reversed(lst)`, `lst + [x]`。
3. **命名避坑守則**：變數名稱加上修飾詞（如 `total_sum`, `num_list`, `max_val`），絕不可直接單用內建函式名。


In [ ]:
# 13.3.6 學生實作練習：安全型態字串拼接與排序過濾器
# 任務說明：實作兩個抗 TypeError 函式
# 1. safe_format_key_value(key, val)：
#    將 key 與 val 組合成 "key = val" 字串。不論傳入的是整數、浮點數、字串甚至是 None，
#    都必須安全轉為字串回傳，絕不引發 TypeError！
# 2. safe_sort_numbers(lst)：
#    傳入串列 lst，回傳一個由小到大排序後的新串列。
#    要求：不得引發 NoneType 賦值錯誤，且原始串列 lst 內容不可被破壞！

def safe_format_key_value(key, val) -> str:
    # 請在此處使用 f-string 實作安全格式化
    return f"{key} = {val}"

def safe_sort_numbers(lst: list) -> list:
    # 請在此處使用回傳新串列的排序方式
    return sorted(lst)

# 測試用例
print("拼接字串與數字:", safe_format_key_value("Score", 100))
print("拼接特殊 None 物件:", safe_format_key_value("Result", None))
original = [5, 2, 8, 1]
sorted_res = safe_sort_numbers(original)
print("排序結果:", sorted_res)
print("原始清單未受污染:", original)


In [ ]:
# 13.3.6 單元測試驗證
assert safe_format_key_value("age", 18) == "age = 18"
assert safe_format_key_value("flag", True) == "flag = True"
assert safe_format_key_value("item", None) == "item = None"

orig = [9, 3, 7]
res = safe_sort_numbers(orig)
assert res == [3, 7, 9]
assert orig == [9, 3, 7] # 確認未被破壞
assert res is not None
print("🎉 13.3.6 所有測試通過！徹底瓦解 TypeError 與 NoneType 幽靈！")


### 13.3.7 RE 排行榜第六名：`RecursionError` 遞迴深度超限（最大堆疊溢出）

在 APCS 實作題中，深度優先搜尋（DFS）、樹狀結構走訪、樹直徑計算以及回溯法（Backtracking）等題目經常仰賴遞迴來解決。然而，許多考生的程式在小測資運作完美，一送出評判卻直接吃下慘烈的 **`RE`**，其罪魁禍首往往是 **`RecursionError: maximum recursion depth exceeded`**！

為什麼會發生 `RecursionError`？
在計算機作業系統與 Python 虛擬機（PVM）底層，每一次發起函式呼叫，系統都必須在呼叫堆疊（Call Stack）中分配一塊記憶體空間（Stack Frame，堆疊訊框），用以儲存該層函式的參數、區域變數與返回位址。如果堆疊層數無止境擴張，就會耗盡系統記憶體導致作業系統強制崩潰（Segmentation Fault）。為了防止直譯器被拖垮，**Python 官方在直譯器內部設定了一道安全保險絲：預設最大遞迴深度限制通常為 1000 層**！

考場三大引爆 `RecursionError` 的核心原因：
1. **基準條件（Base Case）遺漏或無法收斂**：
   - 忘記寫終止條件（如遞迴走到樹葉節點或邊界未 `return`），或者遞迴參數更新方向錯誤（例如本該 `n - 1` 卻寫成 `n + 1`），導致函式如同無窮迴圈般無限自我呼叫，不到 0.01 秒就衝破 1000 層保險絲！
2. **圖論走訪未設「已造訪標記（Visited Flag）」**：
   - 在圖論走訪時，節點 A 呼叫節點 B，節點 B 又回頭呼叫節點 A，在兩點之間形成死循環互叩，瞬間爆棧！
3. **APCS 大測資下的「單鏈退化（Degenerate Chain）」**：
   - 題目給定 $N = 10000$ 筆節點。即便邏輯完全正確、沒有無窮遞迴，但如果輸入資料退化成一條長鏈（例如鏈結串列狀的極端樹），遞迴深度高達 10000 層，直接被 Python 的 1000 層限制當場秒殺！

🛡️ **考場防禦心法：雙管齊下**
1. **考場解禁咒語**：在有深度遞迴的題目開頭，手動調高遞迴上限：
   ```python
   import sys
   sys.setrecursionlimit(200000)
   ```
2. **改寫為迭代與顯式堆疊（Iterative Simulation with Stack）**：
   將系統遞迴改為 `stack = [root]` 搭配 while 迴圈手刻堆疊。Python 的串列建立在 Heap 記憶體上，可以容納數百萬個元素，徹底免疫堆疊溢出！


In [ ]:
# 13.3.7 程式碼演示：無窮遞迴崩潰重現與手刻顯式堆疊防禦
import sys

print(f"Python 當前預設最大遞迴深度: {sys.getrecursionlimit()}")

# 案例 1: 缺少基準條件引爆 RecursionError
def infinite_recursive_call(depth):
    # 故意沒有終止條件
    return infinite_recursive_call(depth + 1)

try:
    infinite_recursive_call(1)
except RecursionError as e:
    print(f"  ❌ 無窮遞迴觸發 RecursionError: {e}")

# 案例 2: 面對超深深度（例如 5000 層），遞迴 vs 手刻迭代堆疊對比
# 傳統遞迴計算累積和
def recursive_sum(n):
    if n <= 1:
        return n
    return n + recursive_sum(n - 1)

# 手刻顯式堆疊迭代計算（永不爆棧！）
def iterative_sum(n):
    total = 0
    while n > 0:
        total += n
        n -= 1
    return total

print("\n--- 深度大測資 (N = 3000) 考驗 ---")
try:
    recursive_sum(3000)
except RecursionError as e:
    print(f"  ❌ recursive_sum(3000) 衝破上限暴斃: {e}")

safe_ans = iterative_sum(3000)
print(f"  🛡️ iterative_sum(3000) 手刻迭代順利完成，結果: {safe_ans}")


### 13.3.7 語法重點回顧與核心觀念提煉

1. **APCS DFS 必打首行咒語**：
   - 只要題目需要寫深度遞迴，程式開頭立刻敲上：
     `import sys; sys.setrecursionlimit(300000)`。
2. **遞迴終止三問自檢**：
   - 我的 Base Case 寫在函式最開頭了嗎？
   - 傳入下一層的參數有確實朝著 Base Case 縮小嗎？
   - 圖論走訪有沒有維護 `visited` 集合防止回頭路？
3. **終極解法**：改用顯式串列模擬堆疊（`stack.append()` / `stack.pop()`），徹底不受遞迴上限束縛。


In [ ]:
# 13.3.7 學生實作練習：抗爆棧費氏數列與階乘計算器
# 任務說明：實作兩個即便面對極大 N 也不會引發 RecursionError 的迭代演算法
# 1. safe_factorial(n)：
#    計算 n 的階乘 (n!)。使用迴圈迭代實作，要求 n=2000 時依然順暢回傳結果，絕不引爆 RecursionError！
# 2. safe_fibonacci(n)：
#    計算費氏數列第 n 項（F(0)=0, F(1)=1, F(2)=1, F(3)=2...）。
#    使用常數空間迭代實作，杜絕暴力遞迴導致的堆疊溢出與超時！

def safe_factorial(n: int) -> int:
    # 請在此處使用迭代實作階乘
    ans = 1
    for i in range(2, n + 1):
        ans *= i
    return ans

def safe_fibonacci(n: int) -> int:
    # 請在此處使用迭代實作費氏數列
    if n <= 0:
        return 0
    if n == 1:
        return 1
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

# 測試用例
print("5! =", safe_factorial(5))
print("Fib(10) =", safe_fibonacci(10))
print("計算 1500! 成功位數:", len(str(safe_factorial(1500))))
print("計算 Fib(500) 成功:", safe_fibonacci(500))


In [ ]:
# 13.3.7 單元測試驗證
assert safe_factorial(0) == 1
assert safe_factorial(1) == 1
assert safe_factorial(6) == 720
# 測試超大 n 是否能免疫 RecursionError
assert safe_factorial(1200) > 0

assert safe_fibonacci(0) == 0
assert safe_fibonacci(1) == 1
assert safe_fibonacci(7) == 13
assert safe_fibonacci(100) == 354224848179261915075
print("🎉 13.3.7 所有測試通過！徹底打破 RecursionError 遞迴堆疊上限枷鎖！")


### 13.3.8 RE 排行榜第七名：`AttributeError` 與 `UnboundLocalError`

在 Python 除錯的世界裡，還有兩種極其陰險、常讓零基礎考生盯著螢幕百思不得其解的 RE 殺手：**`AttributeError`** 與 **`UnboundLocalError`**。

#### 一、`AttributeError`：物件型態與方法嚴重大錯配
- 當你試圖存取某個物件根本未曾擁有的屬性或方法時，直譯器會拋出 `AttributeError: '...' object has no attribute '...'`。
- **高頻手滑情境**：
  1. 對字串呼叫了串列專屬方法：例如想把字元加到字串末尾，卻寫成 `my_str.append('a')`（字串是不可變物件，沒有 append 方法！應寫 `my_str += 'a'`）。
  2. 對串列呼叫了字串專屬方法：例如讀入整行字串陣列後，直接寫 `my_list.split()`（只有字串有 split，串列沒有！）。
  3. 對字典呼叫了 `d.add(x)`（集合才有 add，字典要寫 `d[k] = v`）。

#### 二、`UnboundLocalError`：區域變數「先讀取、後賦值」的時空錯位
- 這是 Python 變數作用域（Scope）最具特色的經典地雷！
- 發生情境：當你在函式內部嘗試修改一個定義在函式外部的全域變數時，例如：
  ```python
  counter = 0
  def update():
      counter += 1 # 實際上等於 counter = counter + 1
  ```
- **直譯器底層解析黑幕**：
  Python 在編譯該函式時，一旦在函式內部發現了 `counter = ...`（賦值語句），編譯器就會強制將 `counter` 判定為**該函式的「區域變數（Local Variable）」**。然而，在執行 `counter + 1` 時，該區域變數尚未被賦予初值，直譯器立刻當頭棒喝拋出：
  `UnboundLocalError: cannot access local variable 'counter' where it is not associated with a value`！

🛡️ **防禦心法**：
1. 方法呼叫前確保型態一致，多用 `isinstance(obj, target_type)` 驗證。
2. 若函式內部確實需要修改全域變數，必須在函式第 1 行宣告 `global var_name`；但在優良的程式設計規範中，更推薦將變數作為參數傳入並透過 `return` 回傳新值！


In [ ]:
# 13.3.8 程式碼演示：AttributeError 型態錯配與 UnboundLocalError 作用域陷阱
print("--- 案例 1: AttributeError 型態方法錯配現場 ---")
test_str = "hello"
try:
    test_str.append(" world") # 字串沒有 append！
except AttributeError as e:
    print(f"  ❌ 字串呼叫 append 觸發 AttributeError: {e}")

test_list = ["a", "b", "c"]
try:
    test_list.split(",") # 串列沒有 split！
except AttributeError as e:
    print(f"  ❌ 串列呼叫 split 觸發 AttributeError: {e}")

print("\n--- 案例 2: UnboundLocalError 區域變數時空錯位現場 ---")
score = 100

def try_add_bonus_buggy():
    # 致命陷阱：函式內出現賦值，score 被視為區域變數；但在加 5 時尚未初始化！
    global_val_attempt = score + 5
    score = global_val_attempt # 這一行導致前一行爆發 UnboundLocalError！

try:
    try_add_bonus_buggy()
except UnboundLocalError as e:
    print(f"  ❌ 未宣告 global 直接修改觸發 UnboundLocalError: {e}")

# 正確解法 1: 使用 global 關鍵字宣告
def add_bonus_with_global():
    global score
    score += 5

add_bonus_with_global()
print(f"  🛡️ 使用 global 修復後 score: {score}")

# 正確解法 2 (更推薦): 參數傳入並回傳，避免全域副作用
def calculate_new_score(current_score):
    return current_score + 10

score = calculate_new_score(score)
print(f"  🛡️ 透過參數傳遞與回傳新 score: {score}")


### 13.3.8 語法重點回顧與核心觀念提煉

1. **容器方法速查口訣**：
   - 串列（List）：`.append()`, `.extend()`, `.pop()`, `.sort()`
   - 字典（Dict）：`.keys()`, `.values()`, `.items()`, `.get()`
   - 集合（Set）：`.add()`, `.discard()`, `.remove()`
   - 字串（Str）：`.split()`, `.join()`, `.replace()`, `.strip()`
2. **函式變數自檢清單**：
   - 在函式內部看到 `x += 1`，先確認 `x` 是函式參數還是全域變數？
   - 若為全域變數，務必補上 `global x`，或改為傳參回傳！


In [ ]:
# 13.3.8 學生實作練習：型態方法相容處理器與安全累加器
# 任務說明：實作兩個抗崩潰函式
# 1. safe_append_or_concat(container, element)：
#    若 container 為串列，使用 .append(element) 加入並回傳 container；
#    若 container 為字串，將 element 轉為字串後以 + 串接並回傳新字串；
#    若為其他型態，安全回傳 None，絕不引發 AttributeError！
# 2. safe_accumulator(values, initial=0)：
#    傳入一串數字 values，在純區域環境中安全累加並回傳總和，絕不依賴或污染全域變數！

def safe_append_or_concat(container, element):
    # 請在此處使用 isinstance 進行型態分支防禦
    if isinstance(container, list):
        container.append(element)
        return container
    elif isinstance(container, str):
        return container + str(element)
    return None

def safe_accumulator(values: list, initial: int = 0) -> int:
    # 請在此處實作獨立的區域累加器
    total = initial
    for v in values:
        total += v
    return total

# 測試用例
my_lst = [1, 2]
print("串列 append:", safe_append_or_concat(my_lst, 3))
print("字串相加:", safe_append_or_concat("Score: ", 95))
print("不支援型態防禦:", safe_append_or_concat(12345, 99))
print("安全累加器:", safe_accumulator([10, 20, 30], 5))


In [ ]:
# 13.3.8 單元測試驗證
l_test = [10, 20]
assert safe_append_or_concat(l_test, 30) == [10, 20, 30]
assert safe_append_or_concat("Hello", " World") == "Hello World"
assert safe_append_or_concat("Num: ", 42) == "Num: 42"
assert safe_append_or_concat({"a": 1}, "b") == None

assert safe_accumulator([1, 2, 3, 4], 0) == 10
assert safe_accumulator([], 100) == 100
print("🎉 13.3.8 所有測試通過！成功化解 AttributeError 與 UnboundLocalError 陷阱！")


### 13.3.9 考場 RE 急救與自測 SOP：常見 7 大 RE 綜合實戰排查演練

在競賽程式設計中，面對冷冰冰的「RE」評判結果，最怕的就是考生像無頭蒼蠅一樣胡亂修改程式碼，甚至把原本寫對的邏輯越改越錯。要在 1 分鐘內精準揪出 RE 並完成止血修復，必須建立一套標準作業程序（SOP）。

📋 **考場 RE 急救 3 步驟 SOP**：
1. **步驟一：看清例外型態（What）**：
   - 本地測試時直奔 Traceback 最末行：是 `IndexError`、`ValueError`、`KeyError` 還是 `ZeroDivisionError`？
   - 若在線上評判系統上（無法直接看到 Traceback），立刻對照題目的**極端資料範圍（Constraints）**：$N=0$ 嗎？字串有空行嗎？分母可能是 0 嗎？
2. **步驟二：鎖定事發座標（Where）**：
   - 找到倒數第二行的出事行號，觀察引發錯誤的運算子或函式（是中括號存取、除法、還是 `int()` 轉換？）。
3. **步驟三：築起條件式護城河（How）**：
   - 競賽程式中，首選「**事前條件檢查（Look Before You Leap, LBYL）**」：
     - 存取串列前：`if 0 <= i < len(a):`
     - 存取字典前：`if k in d:` 或改用 `d.get(k, default)`
     - 進行除法前：`if denom != 0:`
     - 遞迴開頭：確認 `sys.setrecursionlimit()` 與 Base Case。

⚔️ **綜合實戰挑戰**：
下方將給出一套模擬競賽中「千瘡百孔、混入多重 RE 地雷」的考生程式碼，包含空串列越界、無效整數剖析、除以零、字典查無此鍵等連鎖致命缺陷。我們將帶領你一步一步將其重構成堅不可摧的滿分防禦標程！


In [ ]:
# 13.3.9 程式碼演示：考場綜合 RE 地雷排查與修復全流程
print("=== 考場原始充滿 5 大 RE 隱形地雷之代碼 ===")
buggy_records = ["Alice:90", "Bob:invalid", "Charlie:0", "David:85", ""]

def process_competition_scores_buggy(records):
    # 潛在雷區 1: 遇到空字串 records[i] 切割解包直接暴斃
    # 潛在雷區 2: 遇到無效分數 int() 轉型引發 ValueError
    # 潛在雷區 3: 累計不及格人數分母為 0 引發 ZeroDivisionError
    # 潛在雷區 4: 字典未初始化累加引爆 KeyError
    # 潛在雷區 5: 排序後取最高分 lst[0] 面對空串列引發 IndexError
    grade_counts = {}
    valid_scores = []
    
    for r in records:
        name, score_str = r.split(":") # 雷區 1!
        s = int(score_str)             # 雷區 2!
        valid_scores.append(s)
        grade_counts[name] += 1        # 雷區 4!
        
    avg = sum(valid_scores) / len(valid_scores) # 雷區 3!
    best = valid_scores[0]                      # 雷區 5!
    return avg, best

print("--- 啟動考場 1 分鐘 SOP 診斷與全方位安全重構 ---")

def process_competition_scores_bulletproof(records):
    grade_counts = {}
    valid_scores = []
    
    for r in records:
        # [防禦 1] 檢查空字串或格式不合
        parts = r.strip().split(":")
        if len(parts) != 2:
            continue # 安全略過空行或瑕疵資料
        name, score_str = parts[0].strip(), parts[1].strip()
        
        # [防禦 2] 安全數值轉換
        try:
            s = int(score_str)
            valid_scores.append(s)
            # [防禦 3] 字典安全累加
            grade_counts[name] = grade_counts.get(name, 0) + 1
        except ValueError:
            continue # 遇到非數字安全忽略
            
    # [防禦 4] 空資料集與除以零安全守門員
    if not valid_scores:
        return (0.0, None)
        
    avg = sum(valid_scores) / len(valid_scores)
    
    # [防禦 5] 安全取最高分
    best = max(valid_scores)
    return (round(avg, 2), best)

res_avg, res_best = process_competition_scores_bulletproof(buggy_records)
print(f"🎉 重構完成！成功免疫所有 RE，安全計算出平均分: {res_avg}, 最高分: {res_best}")


### 13.3.9 語法重點回顧與核心觀念提煉

考場 RE 零容忍自檢清單（送出前默念 5 條）：
1. **輸入端**：是否有空行？`split()` 後個數是否可能不足？
2. **存取端**：`lst[i]` 的 `i` 有沒有可能等於 `len(lst)`？串列是否可能是空的？
3. **字典端**：是否全部改用 `.get()` 或事先 `in` 檢查？
4. **運算端**：`/`、`//`、`%` 的分母有沒有可能在極端測資下為 0？
5. **遞迴端**：遞迴上限 `sys.setrecursionlimit()` 加了嗎？Base Case 必能觸達嗎？


In [ ]:
# 13.3.9 學生實作練習：考場全方位抗崩潰資料管線
# 任務說明：實作 bulletproof_data_pipeline(raw_records)
# 輸入為一組未清洗的字串清單，每項格式預期為 "姓名,年齡,成績"（例如 "Tom,15,88"）
# 請完成資料處理，回傳一個包含三項指標的字典：
# {
#     "valid_count": 有效學生總人數 (int),
#     "average_score": 有效學生平均成績 (float, 四捨五入至小數點後 1 位；若無有效資料回傳 0.0),
#     "max_score": 有效學生最高成績 (int；若無有效資料回傳 0)
# }
# 要求：面對以下極端狀況必須全數安全防禦，絕不可丟出任何 RE：
# 1. 包含空字串、逗號數量不足 2 個的無效行（防範 IndexError / ValueError）
# 2. 年齡或成績包含非數字字母（防範 ValueError）
# 3. 整份 records 為空串列或全是垃圾資料（防範 ZeroDivisionError / IndexError）

def bulletproof_data_pipeline(raw_records: list) -> dict:
    valid_scores = []
    # 請在此處實作強固型資料管線
    for line in raw_records:
        parts = line.strip().split(",")
        if len(parts) != 3:
            continue
        name, age_str, score_str = parts[0].strip(), parts[1].strip(), parts[2].strip()
        try:
            age = int(age_str)
            score = int(score_str)
            valid_scores.append(score)
        except ValueError:
            continue
            
    if not valid_scores:
        return {"valid_count": 0, "average_score": 0.0, "max_score": 0}
        
    avg = round(sum(valid_scores) / len(valid_scores), 1)
    return {
        "valid_count": len(valid_scores),
        "average_score": avg,
        "max_score": max(valid_scores)
    }

# 測試用例
dirty_input = [
    "Alice,16,90",
    "Bob,17,85",
    "CorruptedLineWithoutComma",
    "Charlie,unknown,95",
    "David,15,not_a_number",
    "",
    "Eve,16,95"
]
print("綜合清洗統計結果:", bulletproof_data_pipeline(dirty_input))
print("空輸入防禦:", bulletproof_data_pipeline([]))


In [ ]:
# 13.3.9 單元測試驗證
dirty_data = [
    "A,15,80",
    "B,16,90",
    "C,invalid,100",
    "D,15,invalid",
    "E,15",
    "   ",
    "F,17,100"
]
res1 = bulletproof_data_pipeline(dirty_data)
assert res1["valid_count"] == 3
assert res1["average_score"] == 90.0
assert res1["max_score"] == 100

# 測試空輸入與全垃圾資料
assert bulletproof_data_pipeline([]) == {"valid_count": 0, "average_score": 0.0, "max_score": 0}
assert bulletproof_data_pipeline(["garbage", ",,,", "a,b,c"]) == {"valid_count": 0, "average_score": 0.0, "max_score": 0}
print("🎉 13.3.9 所有測試通過！恭喜你已修成橫掃全場 Runtime Error 的防禦大師！")


## 13.3 總結與考場除錯防禦全景對照表

在本單元中，我們將 APCS 考場中最致命的 **7 大執行時期錯誤（RE）** 進行了地毯式的剖析，並建立了對應的抗崩潰防守心法：

| RE 錯誤型態 | 核心成因 | 考場高頻引爆點 | 唯一解方與防禦護城河 |
| :--- | :--- | :--- | :--- |
| **`IndexError`** | 索引超出合法邊界 | 0-based 偏誤取 `lst[N]`、相鄰走訪 `a[i+1]`、空串列存取 | `if 0 <= i < len(a):`，善用切片 `lst[i:i+1]` |
| **`ValueError`** | 引數型態合法但數值內容無效 | `int("3.14")`、解包數量不符、`lst.index()` 查無此人 | `int(float(s))`、先檢查 `len(tokens)`、`if x in lst:` |
| **`KeyError`** | 字典查無此鍵或集合移除失敗 | 字典未初始化直接 `+= 1`、`set.remove()` 元素不存在 | `dict.get(k, 0)`、`defaultdict(int)`、`set.discard()` |
| **`ZeroDivisionError`** | 除數或模除數為 0 | 計算平均時未防範空資料集（`len=0`）、動態步長遞減 | 看到 `/`、`//`、`%` 必設非零守門員 `if denom != 0:` |
| **`TypeError`** | 操作了不相容的型態或呼叫非函式 | 字串用 `+` 串接整數、原地方法 `sort()` 賦值成 None | 一律用 `f-string`、區分 `sort()` 與 `sorted()`、嚴禁污染內建名 |
| **`RecursionError`** | 遞迴深度衝破直譯器保險絲 (1000) | 遺漏 Base Case、圖論未設 visited、大測資退化長鏈 | `sys.setrecursionlimit(200000)` 或改寫為手刻迭代堆疊 |
| **`AttributeError` / `UnboundLocalError`** | 型態方法錯配或區域變數先讀後賦值 | 字串呼叫 `.append()`、函式未宣告 `global` 逕行更新 | `isinstance()` 檢查、宣告 `global` 或以參數傳入回傳 |

恭喜你完成了 13-3 的完整訓練！現在你已經具備了看懂所有 Traceback 堆疊並在送出前築起嚴密防禦網的能力。
在下一個單元 **《13-4 例外捕捉語法：try ... except 架構與未知長度輸入處理》** 中，我們將更進一步探討 Python 的原生異常處理機制，掌握 APCS 考場讀取未知長度輸入（EOF）的絕殺通關秘技！
